In [ ]:
# ============================================================
# CLICKTECH HRMS
# FINAL CLEANING + NORMALIZATION PIPELINE
# FINAL OUTPUT = ONLY 8 Tables
# POWER BI + AUTOMATION READY
# ============================================================

# ============================================================
# INSTALL FIRST (RUN ONCE)
# ============================================================

# !pip install openpyxl xlsxwriter

# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np

# ============================================================
# INPUT / OUTPUT
# ============================================================

INPUT_FILE = "/content/Emp-dummy-data.xlsx"
OUTPUT_FILE = "Automation_Employee_Master_Final.xlsx"

# ============================================================
# LOAD RAW FILE
# ============================================================

df = pd.read_excel(INPUT_FILE)

print("=" * 70)
print("RAW DATA LOADED")
print("RAW SHAPE :", df.shape)
print("=" * 70)

# ============================================================
# STANDARDIZE COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

# ============================================================
# REMOVE FULL NULL COLUMNS
# ============================================================

null_cols = [c for c in df.columns if df[c].isna().all()]

df.drop(columns=null_cols, inplace=True)

print("\nFULL NULL COLUMNS REMOVED:")
print(null_cols)

# ============================================================
# REMOVE DUPLICATE / REDUNDANT COLUMNS
# ============================================================

drop_cols = [
    "jobtitle",
    "employername",
    "costcentredescription",
    "esic_account_number",
    "businesstitle",
    "date_of_confirmation",
    "organizationleader",
    "offerlettertype",
    "reportingmanager_name",
    "old_emp_code",
    "process",
    "cscid",
    "csc_id",
    "minimumwag_code"
]

existing_drop_cols = [c for c in drop_cols if c in df.columns]

df.drop(columns=existing_drop_cols, inplace=True)

print("\nREDUNDANT COLUMNS REMOVED:")
print(existing_drop_cols)

# ============================================================
# REMOVE DUPLICATE ROWS
# ============================================================

before = len(df)

df.drop_duplicates(inplace=True)

after = len(df)

print(f"\nDUPLICATE ROWS REMOVED : {before - after}")

# ============================================================
# CLEAN STRINGS
# ============================================================

for col in df.select_dtypes(include="object").columns:

    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .replace("nan", np.nan)
    )

# ============================================================
# DATE CLEANING
# ============================================================

date_keywords = [
    "date",
    "doj",
    "dob",
    "joining",
    "promotion",
    "lwd",
    "confirmation"
]

for col in df.columns:

    if any(k in col for k in date_keywords):

        try:
            df[col] = pd.to_datetime(df[col], errors="coerce")
        except:
            pass

# ============================================================
# AGE CREATION
# ============================================================

if "dob" in df.columns:

    today = pd.Timestamp.today()

    df["age"] = (
        (today - df["dob"]).dt.days / 365
    ).astype("Int64")

# ============================================================
# STATUS STANDARDIZATION
# ============================================================

if "emp_status" in df.columns:

    df["emp_status"] = (
        df["emp_status"]
        .str.upper()
        .replace({
            "NEWJOINER": "NEW JOINER",
            "FNF LOCKED": "FNF LOCKED",
            "FNF INPROC": "FNF IN PROCESS"
        })
    )

# ============================================================
# REMOVE FULLY EMPTY COLUMNS AGAIN
# ============================================================

empty_cols = []

for col in df.columns:

    if df[col].isna().all():
        empty_cols.append(col)

    elif df[col].astype(str).str.strip().eq("").all():
        empty_cols.append(col)

df.drop(columns=empty_cols, inplace=True)

print("\nEMPTY COLUMNS REMOVED:")
print(empty_cols)

# ============================================================
# FINAL CLEANED SHAPE
# ============================================================

print("\nFINAL CLEANED SHAPE :", df.shape)

# ============================================================
# ============================================================
# CREATE FINAL 8 SHEETS
# ============================================================
# ============================================================

# ============================================================
# 1. DEMOGRAPHIC DATA
# ============================================================

demographic_cols = [
    "employee_code",
    "employee_name",
    "emp_status",
    "level",
    "gender",
    "hometown",
    "home_state",
    "present_address",
    "marital_status",
    "father_name",
    "dob",
    "age",
    "bloodgroup",
    "mobile_no",
    "pan",
    "aadhar",
    "permanent_address",
    "personal_email"
]

demographic_cols = [
    c for c in demographic_cols if c in df.columns
]

demographic_df = df[demographic_cols].copy()

# ============================================================
# 2. ACADEMIC & WORK EXPERIENCE
# ============================================================

academic_cols = [
    "employee_code",
    "linkedin_id",
    "total_work_exp",
    "current_role",
    "current_city",
    "bachelors_1",
    "bachelors_2",
    "masters_1",
    "masters_2",
    "doctorate",
    "certification"
]

academic_cols = [
    c for c in academic_cols if c in df.columns
]

academic_df = df[academic_cols].copy()

# ============================================================
# 3. JOINING DATA
# ============================================================

joining_cols = [
    "employee_code",
    "doj",
    "department",
    "designation",
    "official_email",
    "reportingmanager_code",
    "company",
    "location"
]

joining_cols = [
    c for c in joining_cols if c in df.columns
]

joining_df = df[joining_cols].copy()

# ============================================================
# 4. PERFORMANCE & TA
# ============================================================

performance_cols = [
    "employee_code",
    "performance_rating",
    "promotion_date",
    "last_promotion_date",
    "ta1",
    "ta1_date",
    "ta1_stat"
]

performance_cols = [
    c for c in performance_cols if c in df.columns
]

performance_df = df[performance_cols].copy()

# ============================================================
# 5. R&R
# KEEP ONLY IMPORTANT REWARD FIELDS
# ============================================================

rr_cols = [
    "employee_code",
    "r1",
    "given_by_1",
    "r2",
    "given_by_2",
    "r3",
    "given_by_3"
]

rr_cols = [
    c for c in rr_cols if c in df.columns
]

rr_df = df[rr_cols].copy()

# ============================================================
# 6. CHANGE DATA
# KEEP ONLY LATEST ROLE CHANGES
# ============================================================

change_cols = [
    "employee_code",
    "pip",
    "dev_plan",
    "slate",
    "rc1",
    "rc1_date",
    "rc2",
    "rc2_date"
]

change_cols = [
    c for c in change_cols if c in df.columns
]

change_df = df[change_cols].copy()

# ============================================================
# 7. EXIT DATA
# ============================================================

exit_cols = [
    "employee_code",
    "resignation_date",
    "resignation_reason",
    "final_lwd",
    "notice_served",
    "notice_recovered"
]

exit_cols = [
    c for c in exit_cols if c in df.columns
]

exit_df = df[exit_cols].copy()

# ============================================================
# 8. PAYROLL DATA
# ============================================================

payroll_cols = [
    "employee_code",
    "pf_account",
    "bank_name",
    "ifsc",
    "bank_account",
    "base_pay",
    "variable_pay",
    "bonus",
    "salary",
    "ctc",
    "special_allow",
    "hra",
    "conveyance",
    "medical"
]

payroll_cols = [
    c for c in payroll_cols if c in df.columns
]

payroll_df = df[payroll_cols].copy()

# ============================================================
# REMOVE EMPTY COLUMNS FROM EACH SHEET
# ============================================================

def remove_empty_columns(dataframe):

    empty_cols = []

    for col in dataframe.columns:

        if dataframe[col].isna().all():
            empty_cols.append(col)

        elif dataframe[col].astype(str).str.strip().eq("").all():
            empty_cols.append(col)

    dataframe.drop(columns=empty_cols, inplace=True)

    return dataframe

demographic_df = remove_empty_columns(demographic_df)
academic_df = remove_empty_columns(academic_df)
joining_df = remove_empty_columns(joining_df)
performance_df = remove_empty_columns(performance_df)
rr_df = remove_empty_columns(rr_df)
change_df = remove_empty_columns(change_df)
exit_df = remove_empty_columns(exit_df)
payroll_df = remove_empty_columns(payroll_df)

# ============================================================
# EXPORT FINAL EXCEL
# ============================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    demographic_df.to_excel(
        writer,
        sheet_name="1_Demographic_Data",
        index=False
    )

    academic_df.to_excel(
        writer,
        sheet_name="2_Academic_WorkEx",
        index=False
    )

    joining_df.to_excel(
        writer,
        sheet_name="3_Joining_Data",
        index=False
    )

    performance_df.to_excel(
        writer,
        sheet_name="4_Performance_TA",
        index=False
    )

    rr_df.to_excel(
        writer,
        sheet_name="5_RnR",
        index=False
    )

    change_df.to_excel(
        writer,
        sheet_name="6_Change_Data",
        index=False
    )

    exit_df.to_excel(
        writer,
        sheet_name="7_Exit_Data",
        index=False
    )

    payroll_df.to_excel(
        writer,
        sheet_name="8_Payroll_Data",
        index=False
    )

# ============================================================
# SUCCESS MESSAGE
# ============================================================

print("\n" + "=" * 70)
print("FINAL AUTOMATION FILE CREATED SUCCESSFULLY")
print("OUTPUT FILE :", OUTPUT_FILE)
print("=" * 70)

print("\nFINAL SHEETS CREATED:")
print("""
1. Demographic_Data
2. Academic_WorkEx
3. Joining_Data
4. Performance_TA
5. RnR
6. Change_Data
7. Exit_Data
8. Payroll_Data
""")

RAW DATA LOADED
RAW SHAPE : (50, 67)

FULL NULL COLUMNS REMOVED:
['old_emp_code', 'process', 'cscid', 'minimumwag_code', 'csc_id', 'separation_reason', 'separation_type']

REDUNDANT COLUMNS REMOVED:
['jobtitle', 'employername', 'costcentredescription', 'esic_account_number', 'businesstitle', 'date_of_confirmation', 'organizationleader', 'offerlettertype', 'reportingmanager_name']

DUPLICATE ROWS REMOVED : 0

EMPTY COLUMNS REMOVED:
['relieving_date']

FINAL CLEANED SHAPE : (50, 50)

FINAL AUTOMATION FILE CREATED SUCCESSFULLY
OUTPUT FILE : Automation_Employee_Master_Final.xlsx

FINAL SHEETS CREATED:

1. Demographic_Data
2. Academic_WorkEx
3. Joining_Data
4. Performance_TA
5. RnR
6. Change_Data
7. Exit_Data
8. Payroll_Data



/tmp/ipykernel_26038/2837483533.py:134: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")
